# 📊 Notebook 1: Data Verkenning Peilbuis B42C0133

**Doel:** De DINO- en BRO-peilbuisdata inladen, bekijken en visualiseren.

---

## Locatie
- **Peilbuis:** B42C0133 (DINO) / GMW000000069526 (BRO)
- **Coördinaten:** 3.537°E, 51.578°N (bij Axel/Terneuzen, Zeeland)
- **Filters:** 001 en 002 (DINO), buisnummers 1 en 2 (BRO)
- **Periode:** 1995–2004 (ca. 230 metingen per filter, 2x per maand)

---

> 💡 **Tip voor beginners:** Voer elke cel uit met **Shift + Enter**. De uitvoer verschijnt direct onder de cel.

## Stap 1: Importeer benodigde bibliotheken

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# Stel in dat figuren direct in de notebook verschijnen
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print("Bibliotheken geladen!")
print(f"  pandas versie: {pd.__version__}")

## Stap 2: DINO data inladen

De DINO CSV heeft een specieke opzet:
- **Regel 1–12:** Metadata (locatie, filter info, referentie)
- **Regel 13+:** Meetdata (datum, stand t.o.v. MP/MV/NAP)

We gebruiken de kolom **'Stand (cm t.o.v NAP)'** — dat is de grondwaterstand 
ten opzichte van Normaal Amsterdams Peil, omgezet naar meters.

In [ ]:
# Pad naar de data mappen (relatief aan de notebooks/ map)
DATA_DIR = Path("..") / "Mantel_Test"
DINO_DIR = DATA_DIR / "DINO_Grondwaterstanden"
BRO_DIR  = DATA_DIR / "BRO_Grondwatermonitoring" / "BRO_Grondwatermonitoringput" / "GMW000000069526"

def lees_dino_csv(pad: Path) -> pd.Series:
    """
    Leest een DINO grondwaterstandsbestand in.
    
    Stappen:
    1. Sla de eerste 12 metaregels over
    2. Lees de meetdata
    3. Converteer datum en NAP-waarden (cm -> m)
    
    Geeft een pandas Series terug met een DatetimeIndex.
    """
    # Sla metadata-header over (12 regels)
    df = pd.read_csv(
        pad,
        skiprows=12,       # eerste 12 regels zijn metadata
        sep=",",
        quotechar='"',
        encoding="utf-8"
    )
    
    # Kolomnamen opschonen (verwijder aanhalingstekens en spaties)
    df.columns = df.columns.str.strip().str.strip('"')
    
    # Selecteer alleen de datum en NAP-kolom
    datum_kolom = "Peildatum"
    nap_kolom   = "Stand (cm t.o.v NAP)"
    
    # Zet datum om naar datetime
    df[datum_kolom] = pd.to_datetime(df[datum_kolom], format="%d-%m-%Y")
    df = df.set_index(datum_kolom)
    df.index.name = "Datum"
    
    # Converteer cm naar m (NAP)
    gws = df[nap_kolom] / 100.0   # cm → m
    gws.name = pad.stem            # naam = bestandsnaam zonder extensie
    
    # Verwijder eventuele ontbrekende waarden
    gws = gws.dropna()
    
    return gws


# Lees filter 001 en 002
gws_001 = lees_dino_csv(DINO_DIR / "B42C0133001.csv")
gws_002 = lees_dino_csv(DINO_DIR / "B42C0133002.csv")

print(f"Filter 001 geladen: {len(gws_001)} metingen")
print(f"  Periode: {gws_001.index.min().date()} - {gws_001.index.max().date()}")
print(f"  Min/Max: {gws_001.min():.2f} / {gws_001.max():.2f} m NAP")
print()
print(f"Filter 002 geladen: {len(gws_002)} metingen")
print(f"  Periode: {gws_002.index.min().date()} - {gws_002.index.max().date()}")
print(f"  Min/Max: {gws_002.min():.2f} / {gws_002.max():.2f} m NAP")

## Stap 3: BRO data inladen

De BRO GLD (Grondwaterstandonderzoek Landelijke Database) heeft een andere opmaak:
- Waterstand staat in **meters** (niet cm)
- Datum heeft een **tijdzone suffix** (bijv. `+01:00`)

In [ ]:
def lees_bro_csv(pad: Path) -> pd.Series:
    """
    Leest een BRO Grondwaterstandsbestand in.
    Zoekt automatisch de kolomnamenregel (regel met 'tijdstip meting').
    Geeft een pandas Series terug (waterstand in m NAP).
    """
    with open(pad, encoding='utf-8') as f:
        regels = f.readlines()
    header_idx = next((i for i, r in enumerate(regels) if 'tijdstip meting' in r), None)
    if header_idx is None:
        raise ValueError(f'Geen header gevonden in {pad.name}')
    df = pd.read_csv(pad, skiprows=header_idx, sep=',', quotechar='"', encoding='utf-8')
    df.columns = df.columns.str.strip().str.strip('"')
    df = df[df['waterstand'].notna()]
    df['tijdstip meting'] = pd.to_datetime(df['tijdstip meting'], utc=True).dt.tz_localize(None)
    df = df.set_index('tijdstip meting')
    df.index.name = 'Datum'
    df.index = df.index.normalize()
    gws = pd.to_numeric(df['waterstand'], errors='coerce').dropna()
    gws.name = pad.stem
    return gws


# Lees de twee BRO bestanden
bro_73324 = lees_bro_csv(BRO_DIR / "GLD000000073324-full.csv")
bro_75843 = lees_bro_csv(BRO_DIR / "GLD000000075843-full.csv")

print(f"BRO GLD000000073324 geladen: {len(bro_73324)} metingen")
print(f"  Periode: {bro_73324.index.min().date()} tot {bro_73324.index.max().date()}")
print(f"  Min/Max: {bro_73324.min():.2f} / {bro_73324.max():.2f} m NAP")
print()
print(f"BRO GLD000000075843 geladen: {len(bro_75843)} metingen")
print(f"  Periode: {bro_75843.index.min().date()} tot {bro_75843.index.max().date()}")

## Stap 4: DINO vs BRO vergelijken

In [ ]:
# Combineer DINO en BRO in één DataFrame voor vergelijking
vergelijking = pd.DataFrame({
    "DINO filter 001 (m NAP)": gws_001,
    "DINO filter 002 (m NAP)": gws_002,
    "BRO filter 2 (m NAP)":    bro_73324,
})

print("Eerste paar rijen:")
print(vergelijking.head(10).to_string())
print()
print("Beschrijvende statistieken:")
print(vergelijking.describe().round(3).to_string())

## Stap 5: Tijdreeks visualiseren

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Bovenste grafiek: filter 001 DINO vs BRO
ax1.plot(gws_001.index, gws_001.values, 
         color="#1565C0", linewidth=1.5, marker=".", markersize=4,
         label="DINO filter 001")
ax1.plot(bro_73324.index, bro_73324.values, 
         color="#E53935", linewidth=1.5, marker=".", markersize=4, alpha=0.7,
         label="BRO filter 2")
ax1.set_ylabel("Grondwaterstand [m NAP]")
ax1.set_title("Peilbuis B42C0133 — Grondwaterstanden 1995–2004")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f} m"))

# Onderste grafiek: filter 002
ax2.plot(gws_002.index, gws_002.values, 
         color="#2E7D32", linewidth=1.5, marker=".", markersize=4,
         label="DINO filter 002 (dieper)")
ax2.set_ylabel("Grondwaterstand [m NAP]")
ax2.set_xlabel("Datum")
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()

# Opslaan
output_pad = Path("..") / "output" / "tijdreeks_peilbuis_B42C0133.png"
output_pad.parent.mkdir(exist_ok=True)
plt.savefig(output_pad, dpi=150, bbox_inches="tight")
print(f"Figuur opgeslagen: {output_pad}")
plt.show()

## Stap 6: Seizoenspatroon analyseren

Grondwater reageert op neerslag (stijgt) en verdamping (daalt).
We verwachten hogere standen in winter/voorjaar en lagere in zomer/najaar.

In [ ]:
# Maandelijkse gemiddelden
maandgems = gws_001.groupby(gws_001.index.month).mean()
maanden_namen = ["Jan","Feb","Mrt","Apr","Mei","Jun","Jul","Aug","Sep","Okt","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(
    range(1, 13), 
    maandgems.values, 
    color=["#1565C0" if v >= maandgems.mean() else "#F57C00" for v in maandgems.values],
    alpha=0.85,
    edgecolor="white"
)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(maanden_namen)
ax.set_ylabel("Gemiddelde grondwaterstand [m NAP]")
ax.set_title("Seizoenspatroon — Maandgemiddelden DINO filter 001 (1995–2004)")
ax.grid(axis="y", alpha=0.3)

# Referentielijn op gemiddelde
ax.axhline(maandgems.mean(), color="red", linestyle="--", linewidth=1.5, 
           label=f"Gemiddelde: {maandgems.mean():.2f} m")
ax.legend()

plt.tight_layout()
plt.savefig(Path("..") / "output" / "seizoenspatroon_B42C0133.png", dpi=150, bbox_inches="tight")
plt.show()

print("✓ Notebook 1 afgerond!")
print("  Ga verder naar Notebook 02_pastas_model.ipynb voor de tijdreeksanalyse.")